# LLM Decoding and Sampling Strategies Demo (Hugging Face + vLLM)

This notebook demonstrates a variety of decoding strategies for small LLMs using Hugging Face Transformers and acceleration with vLLM:

- Base decoding: greedy, temperature, top-k, top-p
- Advanced methods: typical sampling, tail-free sampling (TFS), repetition penalty, contrastive search
- Speculative decoding (assisted generation) for speedup
- vLLM inference with sampling and speculative decoding

We'll use a small SmolLM family model to keep resource usage low.



In [ ]:
# Environment for week11: Decoding strategies, evaluation harness, LLM-as-a-judge.
# Colab or a fresh virtualenv: run this cell once. Locally: `uv sync --group week11` in the repo root instead.
# Optional, for the vLLM section at the end (Linux + NVIDIA GPU only): %pip install -q "vllm>=0.6"
%pip install -q "torch>=2.4,<3" "transformers>=4.56,<5" "datasets>=3.6,<4" "accelerate>=1.0" "pandas>=2.2" "tqdm>=4.66" "huggingface-hub>=0.25" "lm-eval>=0.4.8" "ipython>=8"

In [ ]:
from typing import Optional, List
import os, math, time, platform, torch
from dataclasses import dataclass

print(f"Python: {platform.python_version()}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()} | Device count: {torch.cuda.device_count()}")

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
DEVICE_MAP = "auto" if torch.cuda.is_available() else None
MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
DRAFT_ID = "HuggingFaceTB/SmolLM2-135M"


Python: 3.12.11
Torch: 2.8.0+cu126
CUDA available: True | Device count: 1


# Why Sampling Matters

Sampling defines how a language model **chooses the next token** from its predicted probability distribution.  
The model gives you logits; everything else is *your problem*:  
temperature, top-k, top-p, nucleus sampling, typical sampling, etc.

## Formal Definition

Given logits vector $ z \in \mathbb{R}^V $:

$$
p_i = \frac{e^{z_i / T}}{\sum_{j=1}^{V} e^{z_j / T}}
$$

where  
- $V$ – vocabulary size  
- $T$ – temperature  
- $p_i$ – probability of token \(i\)

Lower temperature sharpens the distribution; higher makes it flatter.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, set_seed

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
# Ensure pad token for generation
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map=DEVICE_MAP,
)
model.eval()

prompt = """Continue the story:\nOnce upon a time"""
set_seed(421)

## Top-K vs Top-P (Nucleus) Sampling

### Top-K
We keep only the **K most probable tokens**, re-normalize, draw a token.

Pros: stable, predictable.  
Cons: loses adaptivity (context may give long tail where you *need* a rare token).

### Top-P (Nucleus)
We choose the **smallest set of tokens whose cumulative probability ≥ P**.

Pros: adaptive; model can use more diversity when distribution is flat.  
Cons: unpredictable size of candidate set.

$$
\sum_{i \in S} p_i \ge P,\quad S = \text{minimal subset}
$$


In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class SampleConfig:
    max_new_tokens: int = 128
    temperature: Optional[float] = None
    top_k: Optional[int] = None
    top_p: Optional[float] = None
    typical_p: Optional[float] = None  # typical sampling
    repetition_penalty: Optional[float] = None
    do_sample: bool = True


def generate_once(prompt: str, cfg: SampleConfig, print_time: bool = True) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen_kwargs = dict(
        max_new_tokens=cfg.max_new_tokens,
        do_sample=cfg.do_sample,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Pass decoding parameters directly to generate / GenerationConfig
    if cfg.temperature is not None:
        gen_kwargs["temperature"] = cfg.temperature
    if cfg.top_k is not None:
        gen_kwargs["top_k"] = cfg.top_k
    if cfg.top_p is not None:
        gen_kwargs["top_p"] = cfg.top_p
    if cfg.typical_p is not None:
        gen_kwargs["typical_p"] = cfg.typical_p
    if cfg.repetition_penalty is not None:
        gen_kwargs["repetition_penalty"] = cfg.repetition_penalty

    start = time.time()
    with torch.no_grad():
        out_ids = model.generate(**inputs, **gen_kwargs)
    took = time.time() - start
    if print_time:
        print(f"Generated {out_ids.shape[-1] - inputs['input_ids'].shape[-1]} tokens in {took:.2f}s")
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)


## A Few Examples

In [ ]:
# Base decoding demos
print("--- Greedy (argmax) ---")
print(generate_once(prompt, SampleConfig(max_new_tokens=96, do_sample=False)))


--- Greedy (argmax) ---
Generated 96 tokens in 4.18s
Continue the story:
Once upon a time, there was a young girl named Lily. She lived in a big house with her family. One day, Lily's mom, Mrs. Jenkins, decided to take her to the library to read a book. Mrs. Jenkins was very kind and helpful, and she made sure that Lily had a good time.

Lily's mom, Mrs. Jenkins, was very kind and helpful. She made sure that Lily had a good time.

One day, Lily's mom


In [ ]:

print("\n--- Temperature=0.7 (soften distribution) ---")
print(generate_once(prompt, SampleConfig(max_new_tokens=96, temperature=0.7)))




--- Temperature=0.7 (soften distribution) ---
Generated 96 tokens in 3.83s
Continue the story:
Once upon a time, there was a boy named Timmy. Timmy loved to play with his friends in the park. The day was really hot and sunny, so Timmy decided to go out and play some games.

He started by building a big pile of sand and some small balls. He put some sand in one of the buckets and a small ball in the other. Next, he put some sand in the buckets together, and then he put a rock in the bucket of sand with the large


In [ ]:

print("\n--- Top-k=40 ---")
print(generate_once(prompt, SampleConfig(max_new_tokens=96, top_k=40)))



--- Top-k=40 ---
Generated 96 tokens in 4.09s
Continue the story:
Once upon a time, there lived a beautiful fairy who lived in a magical garden on the outskirts of a big kingdom. Every night, the fairy would come to the garden with a big, happy smile. She would climb up the trees leading up to the kingdom, where she would sing fairy stories to all the animals.
Once the fairy was in the kingdom, there was a little ball of ice. Every time the fairy came to the fairy ball, one of the enchanted cookies in the ball


In [ ]:

print("\n--- Top-p=0.9 (nucleus) ---")
print(generate_once(prompt, SampleConfig(max_new_tokens=96, top_p=0.9)))


--- Top-p=0.9 (nucleus) ---
Generated 96 tokens in 3.65s
Continue the story:
Once upon a time in the town of Ravenswood, there lived a young woman named Lyra. She was always curious and loved reading books.

One summer, Lyra and her friends decided to go on a trip to the nearby woods. They spent their days exploring the woods and learning about nature. But when they got back home, they found something unusual - a book of fairy tales in the attic of the old mansion.

The tree with the book in it was a famous


## Advanced decoding
---
### Typical Sampling

Typical Sampling attempts to select tokens whose **surprise** (say "Hi!" to the Fedor's seminar group) (self-information) is close to the *expected* entropy of the model’s current distribution.

Instead of truncating by probability mass (Top-P) or by rank (Top-K), typical sampling selects tokens that minimize the deviation:

$$
| -\log p_i - H(p) |
$$

where:

- $p_i$ — probability of token $i$  
- $H(p)$ — entropy of the distribution  
  $$
  H(p) = - \sum_j p_j \log p_j
  $$

#### Intuition

- Tokens whose information content is *too low* are overly predictable and may make text boring.  
- Tokens whose information content is *too high* are surprising outliers and often lead to hallucinations.  
- Typical sampling keeps tokens whose “surprise level” matches the “natural” uncertainty of the model.

#### Procedure

1. Compute token probabilities $p_i$.  
2. Compute entropy $H(p)$.  
3. Compute the information deviation score  
   $$
   d_i = | -\log p_i - H(p) |.
   $$  
4. Sort tokens by $d_i$ (ascending).  
5. Keep top tokens until cumulative probability exceeds a threshold (e.g., 0.95).  
6. Sample from that filtered set.

This method is more stable than Top-P when the distribution is skewed and more adaptive than Top-K in open-ended decoding.

---

## Repetition Penalty

Repetition penalty discourages the model from generating tokens that have already appeared in the context by **penalizing their logits**.

If a token $t$ has been generated before, its logit $z_t$ is transformed as:

$$
z_t' =
\begin{cases}
\frac{z_t}{\text{penalty}}, & z_t > 0 \\
z_t \cdot \text{penalty}, & z_t < 0
\end{cases}
$$

where:

- $\text{penalty} > 1$  
  (typical values: 1.1–2.0)

#### Effect

- If the model “really wants” to repeat a word (large positive logit), dividing reduces its relative strength.  
- If the model assigns negative evidence to a word, multiplying makes it even less likely.  
- The penalty grows multiplicatively, so repeated tokens become exponentially less attractive over time.

#### Why It Helps

Without repetition control, models often enter degenerate loops:

> “I think that I think that I think…”

The repetition penalty counteracts this by flattening the logits of previously used tokens.

#### Relation to Frequency Penalties

Some systems (e.g., OpenAI APIs) use *frequency* and *presence* penalties instead, which operate as:

$$
z_t' = z_t - \alpha \cdot \text{count}(t),
$$

but the multiplicative penalty above is more effective at preventing hard loops and works better with large vocabularies.

---



In [ ]:
print("--- Typical sampling (p=0.2) ---")
print(generate_once(prompt, SampleConfig(max_new_tokens=96, typical_p=0.2)))


--- Typical sampling (p=0.2) ---
Generated 96 tokens in 3.70s
Continue the story:
Once upon a time, in a world with no borders, there was a big disagreement about how to deal with the mysterious black dog that was seen lurking around town. The dog had been watching the people and was making sure no one saw or heard anything that might cause trouble. The dog had even tried to steal some food from a shop on the outskirts of town, but nobody believed him.

However, the owner of the shop had found a special box that seemed to contain some very useful things


In [ ]:

print("\n--- Repetition penalty=1.2 + top-p=0.9 ---")
print(generate_once(prompt, SampleConfig(max_new_tokens=96, top_p=0.9, repetition_penalty=1.2)))



--- Repetition penalty=1.2 + top-p=0.9 ---
Generated 96 tokens in 4.18s
Continue the story:
Once upon a time, in a faraway land called Africa where many people lived like humans but were different from them than we are. They had unique shaped of skin and hair that looked very much like us even though they didn't have their own way of life to live because it was all part of our world being one large living planet which is called Earth. In this ancient earth an old man named Akua thought about how big it's gonna be when the coming humans arrive at earth so he knew there


# Advanced Decoding Methods

## Contrastive Search

Contrastive Search balances *fluency* and *relevance* by maximizing:

### Theoretical Objective

$$
\text{score}(x_t) =
\lambda \cdot \log p(x_t \mid x_{<t})
-
(1 - \lambda) \cdot \max_{j < t} \cos(h_t, h_j)
$$

- **Fluency term** encourages selecting tokens with high conditional probability.  
- **Degeneration penalty** discourages tokens whose hidden states resemble earlier ones (prevents loops, blandness, and repetitive thinking).  
- The parameter $\lambda$ controls the tradeoff between the two forces.

This leads to more **focused**, **coherent**, and **less repetitive** generation compared to top-k or top-p sampling.

---

### Hugging Face Implementation

Hugging Face exposes Contrastive Search through:

- `penalty_alpha` — strength of the contrastive penalty  
- `top_k` — restricts candidate tokens before contrast is applied  

In practice, the implementation proceeds as:

1. Compute logits for the next token.
2. Select the **top_k** candidate tokens by logit value.
3. For each candidate, compute a contrastive penalty:

   $$
   \alpha \cdot \max_{j < t} \cos(h_t, h_j)
   $$

   where  
   - $h_t$ = hidden state for candidate continuation,  
   - $h_j$ = hidden states of previously generated tokens,  
   - $\alpha = \text{penalty_alpha}$.

4. Adjust each candidate’s logit:

   $$
   \text{adjusted_logit}
   =
   \text{logit}
   -
   \alpha \cdot \max \cos(h_t, h_j)
   $$

5. Select the token with the **highest adjusted logit** (no sampling).

This simplifies the theoretical form into two intuitive knobs:

- **`top_k`** – controls fluency (using only the top-K most likely tokens),  
- **`penalty_alpha`** – controls relevance and anti-degeneration strength.

Together, they approximate the theoretical Contrastive Search objective efficiently and at scale.

---


In [ ]:
# https://huggingface.co/transformers-community/contrastive-search
print("\n--- Contrastive search (penalty_alpha=0.6, top_k=4) ---")
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=96,
        do_sample=False,
        penalty_alpha=0.6,
        top_k=4,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Contrastive search was moved to a `custom_generate` repo: https://hf.co/transformers-community/contrastive-search. To prevent loss of backward compatibility, add `custom_generate='transformers-community/contrastive-search'` to your `generate` call before v4.62.0.
Contrastive search requires `trust_remote_code=True` in your `generate` call, since it loads https://hf.co/transformers-community/contrastive-search.



--- Contrastive search (penalty_alpha=0.6, top_k=4) ---
The repository `transformers-community/contrastive-search` contains custom generation code that will override the default `generate` method. You can inspect the repository content at https://hf.co/transformers-community/contrastive-search.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
Continue the story:
Once upon a time, there was a man named John who lived in a small town. One day, John's wife, Sarah, died in a car accident. The news spread quickly, and people started to worry about John's health.

John's wife, Sarah, was a kind and gentle person, but she couldn't bear to see her husband's death. She felt a deep sadness and regret for not being able to care for him as much as he needed to.

John



## DoLa (Decoding by Contrasting Layers)

DoLa contrasts **shallow vs. deeper transformer layers**:

- Shallow layers produce generic, frequent, “boring” next-token guesses.  
- Deep layers produce more specific, contextualized continuations.

The core idea is to compute:

$$
\Delta z = z_{\text{deep}} - z_{\text{shallow}}
$$

and decode from the **contrastive** logits $\Delta z$ instead of the raw logits.

Effects:

- Suppresses generic filler text produced by early layers.  
- Enhances specific, context-aware reasoning from deeper layers.  
- Produces sharper, more meaningful, non-generic generations.
---
## How Hugging Face Implements DoLa in Practice

DoLa is not part of the standard `transformers` library.  
It becomes available only when:

- the model implements a custom `generate()` override, and  
- you enable it via `trust_remote_code=True`.

In these implementations, DoLa works as follows:

### 1. Forward Pass Through All Layers

For each decoding step, the model computes:

- **shallow-layer logits**  
  (e.g., from the first L layers)

- **deep-layer logits**  
  (e.g., the final decoder layer)

Let:

- $z_{\text{deep}}$ — logits from the final transformer block  
- $z_{\text{shallow}}$ — logits averaged or pooled from early blocks

### 2. Layer Selection Mode

The argument:

```
dola_layers="low"
```

directs the model to use only the *shallow* transformer blocks as the baseline for contrast.

Other implementations support:

- `"high"` — contrast against deeper mid-layers  
- `"all"` — contrast against a span of layers  

(Exact modes depend on the model’s custom code.)

### 3. Contrastive Logit Construction

The generation head receives **contrastive logits**:

$$
\Delta z = z_{\text{deep}} - z_{\text{shallow}}
$$

This removes generic, high-frequency predictions (high in shallow layers) and highlights context-specific predictions (emerging in deep layers).

### 4. Deterministic Decoding

Most DoLa implementations:

- disable sampling (`do_sample=False`),  
- select the token with maximal $\Delta z$,  
- continue the forward pass with updated KV-cache.

This is computationally lightweight because the shallow and deep activations are already produced during the same forward pass.

### Practical Effect

- Shallow layers contribute “autopilot” language modeling.  
- Deep layers inject semantic specificity.  
- Subtracting shallow from deep removes clichés while preserving coherence.

The result is sharper, less generic, more semantically grounded generation.


In [ ]:
print("\n--- DoLa (on low layers) ---")
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=96,
        do_sample=False,
        dola_layers="low",
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        trust_remote_code=True,
    )
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))


DoLa generation was moved to a `custom_generate` repo: https://hf.co/transformers-community/dola. To prevent loss of backward compatibility, add `custom_generate='transformers-community/dola'` to your `generate` call before v4.62.0.



--- DoLa (on low layers) ---


Continue the story:
Once upon a time in a land filled with magical creatures and mysterious events, a young boy named Max discovers he has the ability to control time. Max is 12 and a half and is in high school. Max is 11 and is 11 and 1. Max is 11 and 1 and 1 and 1 and 1 and 1 and 1 and 1 and 1 and 1 and 1 and 1 and 1 and 


### Speculative Decoding (Assisted Generation)

Speculative decoding accelerates generation by pairing a **large target model** with a **smaller draft model**:

- The **draft model** proposes several next tokens quickly.  
- The **target model** verifies or rejects them in a single forward pass.  
- Verified tokens are accepted without recomputing intermediate steps.  
- Rejected tokens trigger a correction step where the target model regenerates from the last valid prefix.

This reduces the number of expensive forward passes through the large model and substantially speeds up decoding while preserving output quality.

---

#### How It Works

Let:

- **Large model** = expensive, accurate, final authority  
- **Draft model** = small, fast, approximate predictor  

The algorithm proceeds as follows:

1. Run the **draft model** to produce a block of $k$ speculative tokens.  
2. Run the **large model** once on the *current prefix* and compute its true probability distribution.  
3. For each draft token:
   - If the large model’s probability agrees (above a threshold), **accept** it.  
   - If not, **reject** and let the large model regenerate from that point.  
4. Continue with the newly extended prefix.

This keeps the large model’s output **exact**, while delegating most of the cheap proposals to the small model.

---

#### Hugging Face Implementation

Hugging Face exposes speculative decoding with the argument:

```
assistant_model=draft_model
```

Inside `generate()`:

- The draft model is called first to predict multiple candidate tokens.  
- The main model verifies them in batched forward passes.  
- Only the large model’s verified outputs appear in the final sequence.  
- No sampling is required (`do_sample=False` is typical).  

Because both models live inside the generation loop, the API behaves like standard decoding but runs **significantly faster**, especially on long sequences.

---

#### Practical Notes

- The draft model **must** be different from the target model (hence the safeguard in the example).  
- Speedup grows with:
  - draft model being much smaller  
  - long output lengths  
  - deterministic decoding  
- Output quality remains equivalent to regular greedy decoding from the large model, because speculative tokens that differ from the target model’s predictions are rejected.

Speculative decoding therefore provides a **strict acceleration** without changing the semantics or correctness of the model’s output.


In [ ]:
from transformers import AutoModelForCausalLM, set_seed

# Load draft model for assisted generation
if DRAFT_ID == MODEL_ID:
    DRAFT_ID = "HuggingFaceTB/SmolLM2-135M"  # ensure a distinct draft if user overrides env

draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_ID,
    dtype=DTYPE,
    device_map=DEVICE_MAP,
)
draft_model.eval()

BIG_MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B"
model = AutoModelForCausalLM.from_pretrained(
    BIG_MODEL_ID,
    dtype=DTYPE,
    device_map=DEVICE_MAP,
)

tokenizer = AutoTokenizer.from_pretrained(BIG_MODEL_ID, use_fast=True)
model.eval()

set_seed(43)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
start = time.time()
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))
print(f"Base decoding time: {time.time() - start:.2f}s")

set_seed(43)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
start = time.time()
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        assistant_model=draft_model,  # pass draft directly
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))
print(f"Assisted decoding time: {time.time() - start:.2f}s")


In [ ]:

set_seed(43)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
start = time.time()
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))
print(f"Base decoding time: {time.time() - start:.2f}s")

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Continue the story:
Once upon a time, there was a little girl named Lily. She lived in a cozy house with her family, and she loved to play outside. One day, while playing in the backyard, Lily found a strange object. It was a small, shiny box with buttons and a screen. She didn't know what it was, but she was curious.

Lily asked her mom, "What is this thing?"

Her mom replied, "That's called a computer, sweetheart. It's a tool that people use to learn new things, talk to friends, and even play games."

Lily was fascinated. She wanted to learn more
Base decoding time: 101.38s


In [ ]:

set_seed(43)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
start = time.time()
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        assistant_model=draft_model,  # pass draft directly
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))
print(f"Assisted decoding time: {time.time() - start:.2f}s")

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for o

Continue the story:
Once upon a time, there was a little girl named Lily. She loved to play outside and explore her neighborhood. One day, while playing near the park, she saw a group of kids playing a game called "tag." They were running around, laughing, and having fun. Lily wanted to join in, but she didn't know how.

She asked her mom, "What is tag?" Her mom explained, "Tag is a game where one person is 'it,' and they try to touch someone else without getting caught. When they do, they become 'it' too!"

Lily thought that sounded like a lot of fun! But
Assisted decoding time: 40.79s


### vLLM Acceleration

vLLM provides a highly optimized inference engine that significantly speeds up decoding compared to standard Hugging Face generation. Its main performance gains come from two core innovations:

---

#### PagedAttention

PagedAttention is a memory-efficient KV-cache management system inspired by OS virtual memory.  
Instead of storing attention KV tensors as large, contiguous blocks, vLLM organizes them into **fixed-size pages**:

- avoids GPU memory fragmentation  
- enables efficient reuse of cached tokens  
- allows serving many requests concurrently  
- makes long-context inference drastically cheaper  

Because attention reads only need to access the relevant pages, the model avoids most of the overhead incurred by standard transformers.

This allows vLLM to serve many generations in parallel without running out of memory.

---

#### Continuous Batching

vLLM dynamically schedules incoming requests into a **single continuously updating batch**:

- late requests join the current batch  
- completed sequences drop out  
- GPU utilization remains near optimal  
- throughput increases dramatically under real workloads  

Unlike static batching (fixed batch created once), continuous batching ensures the GPU is always busy.

---

#### Basic Text Generation in vLLM

The first part of the example runs standard vLLM generation using:

- `LLM(model=...)` — loads the model with vLLM’s optimized runtime  
- `SamplingParams(...)` — configures generation options  
- `llm.generate([...], sampling_params=...)` — executes optimized decoding  

This already outperforms conventional Hugging Face decoding.

---

#### Prefix Caching

For repeated prompts with long shared prefixes, vLLM automatically **reuses the KV-cache of the prefix**:

- Only the *new* suffix tokens require computation.  
- Queries with identical or overlapping prefixes skip most of the forward pass.  
- This produces large speedups for retrieval, question answering, and structured tasks.

In the second half of the example, two queries share the large `LONG_PROMPT`.  
vLLM computes this prefix **once**—the second query becomes substantially faster because prefix caching eliminates redundant KV-cache computation.



In [ ]:
import gc
import torch

# free the GPU before vLLM takes it
for name in ["model", "draft_model"]:
    if name in globals():
        del globals()[name]
torch.cuda.empty_cache()
gc.collect()

In [ ]:
import time
from transformers import set_seed

In [ ]:
BIG_MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
DRAFT_ID = "HuggingFaceTB/SmolLM2-135M"
prompt = """Continue the story:\nOnce upon a time"""
set_seed(421)

> **Requirements.** vLLM runs on Linux with an NVIDIA GPU (Colab T4 is fine). On macOS or without a GPU, stop here: everything above does not need it.

In [ ]:
from vllm import LLM, SamplingParams

# vLLM basic text generation
vllm_model = BIG_MODEL_ID
llm = LLM(model=vllm_model, max_model_len=2048,)
start = time.time()
sampling = SamplingParams(
    max_tokens=128,
    temperature=0.7,
    top_p=0.9,
    top_k=40,
)

outputs = llm.generate([prompt], sampling)
print(outputs[0].outputs[0].text)
print(f"Base decoding time: {time.time() - start:.2f}s")


In [ ]:
LONG_PROMPT = (
    "You are a helpful assistant in recognizes the content of tables in markdown format. Here is a table as follows.\n# Table\n"
    + """
| ID  | Name          | Age | Occupation    | Country       | Email                  | Phone Number   | Address                       |
|-----|---------------|-----|---------------|---------------|------------------------|----------------|------------------------------|
| 1   | John Doe      | 29  | Engineer      | USA           | john.doe@example.com   | 555-1234       | 123 Elm St, Springfield, IL  |
| 2   | Jane Smith    | 34  | Doctor        | Canada        | jane.smith@example.com | 555-5678       | 456 Oak St, Toronto, ON      |
| 3   | Alice Johnson | 27  | Teacher       | UK            | alice.j@example.com    | 555-8765       | 789 Pine St, London, UK      |
| 4   | Bob Brown     | 45  | Artist        | Australia     | bob.b@example.com      | 555-4321       | 321 Maple St, Sydney, NSW    |
| 5   | Carol White   | 31  | Scientist     | New Zealand   | carol.w@example.com    | 555-6789       | 654 Birch St, Wellington, NZ |
| 6   | Dave Green    | 28  | Lawyer        | Ireland       | dave.g@example.com     | 555-3456       | 987 Cedar St, Dublin, IE     |
| 7   | Emma Black    | 40  | Musician      | USA           | emma.b@example.com     | 555-1111       | 246 Ash St, New York, NY     |
| 8   | Frank Blue    | 37  | Chef          | Canada        | frank.b@example.com    | 555-2222       | 135 Spruce St, Vancouver, BC |
| 9   | Grace Yellow  | 50  | Engineer      | UK            | grace.y@example.com    | 555-3333       | 864 Fir St, Manchester, UK   |
| 10  | Henry Violet  | 32  | Artist        | Australia     | henry.v@example.com    | 555-4444       | 753 Willow St, Melbourne, VIC|
| 11  | Irene Orange  | 26  | Scientist     | New Zealand   | irene.o@example.com    | 555-5555       | 912 Poplar St, Auckland, NZ  |
| 12  | Jack Indigo   | 38  | Teacher       | Ireland       | jack.i@example.com     | 555-6666       | 159 Elm St, Cork, IE         |
| 13  | Karen Red     | 41  | Lawyer        | USA           | karen.r@example.com    | 555-7777       | 357 Cedar St, Boston, MA     |
| 14  | Leo Brown     | 30  | Chef          | Canada        | leo.b@example.com      | 555-8888       | 246 Oak St, Calgary, AB      |
| 15  | Mia Green     | 33  | Musician      | UK            | mia.g@example.com      | 555-9999       | 975 Pine St, Edinburgh, UK   |
| 16  | Noah Yellow   | 29  | Doctor        | Australia     | noah.y@example.com     | 555-0000       | 864 Birch St, Brisbane, QLD  |
| 17  | Olivia Blue   | 35  | Engineer      | New Zealand   | olivia.b@example.com   | 555-1212       | 753 Maple St, Hamilton, NZ   |
| 18  | Peter Black   | 42  | Artist        | Ireland       | peter.b@example.com    | 555-3434       | 912 Fir St, Limerick, IE     |
| 19  | Quinn White   | 28  | Scientist     | USA           | quinn.w@example.com    | 555-5656       | 159 Willow St, Seattle, WA   |
| 20  | Rachel Red    | 31  | Teacher       | Canada        | rachel.r@example.com   | 555-7878       | 357 Poplar St, Ottawa, ON    |
| 21  | Steve Green   | 44  | Lawyer        | UK            | steve.g@example.com    | 555-9090       | 753 Elm St, Birmingham, UK   |
| 22  | Tina Blue     | 36  | Musician      | Australia     | tina.b@example.com     | 555-1213       | 864 Cedar St, Perth, WA      |
| 23  | Umar Black    | 39  | Chef          | New Zealand   | umar.b@example.com     | 555-3435       | 975 Spruce St, Christchurch, NZ|
| 24  | Victor Yellow | 43  | Engineer      | Ireland       | victor.y@example.com   | 555-5657       | 246 Willow St, Galway, IE    |
| 25  | Wendy Orange  | 27  | Artist        | USA           | wendy.o@example.com    | 555-7879       | 135 Elm St, Denver, CO       |
| 26  | Xavier Green  | 34  | Scientist     | Canada        | xavier.g@example.com   | 555-9091       | 357 Oak St, Montreal, QC     |
| 27  | Yara Red      | 41  | Teacher       | UK            | yara.r@example.com     | 555-1214       | 975 Pine St, Leeds, UK       |
| 28  | Zack Blue     | 30  | Lawyer        | Australia     | zack.b@example.com     | 555-3436       | 135 Birch St, Adelaide, SA   |
| 29  | Amy White     | 33  | Musician      | New Zealand   | amy.w@example.com      | 555-5658       | 159 Maple St, Wellington, NZ |
| 30  | Ben Black     | 38  | Chef          | Ireland       | ben.b@example.com      | 555-7870       | 246 Fir St, Waterford, IE    |
"""
)

In [ ]:
from vllm import LLM, SamplingParams

In [ ]:
def get_generation_time(llm, sampling_params, prompts):
    # time the generation
    start_time = time.time()
    output = llm.generate(prompts, sampling_params=sampling_params)
    end_time = time.time()
    # print the output and generation time
    print("-" * 30)
    print(f"Output: {output[0].outputs[0].text}")
    print(f"Generation time: {end_time - start_time} seconds.")
    print("-" * 30)

In [ ]:
llm = LLM(model="HuggingFaceTB/SmolLM2-360M", enable_prefix_caching=True)

sampling_params = SamplingParams(temperature=0, max_tokens=100)

# Querying the age of John Doe
get_generation_time(
    llm,
    sampling_params,
    LONG_PROMPT
    + "Question: what is the age of John Doe? Your answer: The age of John Doe is ",
)

# Querying the age of Zack Blue
# This query will be faster since vllm avoids computing the KV cache of LONG_PROMPT again.
get_generation_time(
    llm,
    sampling_params,
    LONG_PROMPT
    + "Question: what is the age of Zack Blue? Your answer: The age of Zack Blue is ",
)
